# ChildBEx PR11 — EfficientNetV2B0 training (thin orchestration)

All training logic lives in the `childbex_ml.training` package; this notebook only
checks the runtime, verifies the PR10.5 cloud artifact and calls the package API.

Rules:
- Input is **only** a PR10.5 cloud artifact (privacy-minimized tensors; no DICOM). Make it
  available manually (e.g. copy it to `/content/`); this notebook does not upload, sync or download data.
- Real-patient artifacts may only be used here under the ChildBEx data-governance decision for external training.
- **TEST is never used in PR11**: it is only covered by the artifact's structural shard verification.
- Outputs are aggregate counts and metrics only. Do not print samples, tokens or tensors.

## 1. Runtime and device (information only — the Colab default is not trusted)

In [ ]:
import platform, subprocess
print("python", platform.python_version())
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("no NVIDIA GPU visible")

## 2. Install the exact ChildBEx ML wheel and TRAINING_RUNTIME_V1

Upload the locally built wheel (`childbex_ml-0.2.0-py3-none-any.whl`, see `apps/ml/README.md`) first.
If NumPy, TensorFlow or Keras changed, **restart the runtime** (Runtime → Restart session) and continue with step 3.

In [ ]:
WHEEL = "/content/childbex_ml-0.2.0-py3-none-any.whl"
%pip install --quiet "{WHEEL}[train]" "tensorflow==2.20.0" "keras==3.11.3" "numpy==2.1.3"

## 3. Verify TRAINING_RUNTIME_V1 (fails with UNSUPPORTED_RUNTIME otherwise)

In [ ]:
from childbex_ml.training import check_runtime
from childbex_ml.training.runtime import describe_runtime
check_runtime()
runtime = describe_runtime()
print({k: runtime[k] for k in ("runtimeProfile", "pythonVersion", "tensorflowVersion", "kerasVersion", "numpyVersion")})
print(runtime["tensorflowBuild"], [d["type"] for d in runtime["devices"]])

## 4. Verify the PR10.5 cloud artifact (aggregates only; no TEST counts)

In [ ]:
ARTIFACT_DIR = "/content/cloud-artifact"   # set manually
RUN_DIR = "/content/training-runs/run-001"  # must not exist yet
PRESET = "efficientnetv2b0-baseline-v1"

from childbex_ml.cloud_artifact import CloudArtifact
from childbex_ml.training.train import artifact_binding, check_splits

artifact = CloudArtifact.open(ARTIFACT_DIR)  # marker, manifest, artifactSha256, exact file set, every shard SHA-256
binding = artifact_binding(artifact)          # 224x224x3 float32 tensor contract
counts = check_splits(artifact)               # TRAIN / VALIDATION only
print("artifactSha256", binding["artifactSha256"])
print("preprocessingConfigHash", binding["preprocessingConfigHash"])
print("TRAIN", counts["TRAIN"], "| VALIDATION", counts["VALIDATION"])

## 5. Train (HEAD phase, then FINE_TUNE) through the package API

In [ ]:
from childbex_ml.training import load_preset
from childbex_ml.training.train import train

config = load_preset(PRESET)   # or load_config_file("/content/my-training-config.json")
summary = train(ARTIFACT_DIR, config, RUN_DIR, verbose=2)
print("trainingConfigSha256", summary["trainingConfigSha256"])
print("trainingRecipeSha256", summary["trainingRecipeSha256"])
for phase, info in summary["phases"].items():
    print(phase, "best epoch", info["bestEpoch"], info["best"])

## 6. Validation curves (aggregate history only)

In [ ]:
import json, matplotlib.pyplot as plt
history = json.load(open(f"{RUN_DIR}/history.json"))["epochs"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ("loss", "pr_auc", "roc_auc")):
    ax.plot([h["train"][metric] for h in history], label=f"train {metric}")
    ax.plot([h["validation"][f"val_{metric}"] for h in history], label=f"val_{metric}")
    ax.set_xlabel("epoch (HEAD then FINE_TUNE)"); ax.legend()
plt.show()

The run directory (`RUN_COMPLETE.json` written last) is portable: copy it back to the
controlled environment for PR12 evaluation. A directory without `RUN_COMPLETE.json` is incomplete.